In [4]:
"""
f = min( x1 + x4 + max(x3, x2),  x3 + x2 + max(x1, x4) )
Known x values substituted as constants. Variables: t1, t2 (integer), b (binary).
"""
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

x1, x2, x3, x4 = 6, 7, 1, 9
M = 2                                  # penalty weight (> 1)

A = x1 + x4                            # branch 0 base: A + max(x2, x3)
B = x2 + x3                            # branch 1 base: B + max(x1, x4)

qp = QuadraticProgram("minmax_v2")
qp.integer_var(lowerbound=0, upperbound=x2 + x3, name="t1")   # t1 ~ max(x2, x3)
qp.integer_var(lowerbound=0, upperbound=x1 + x4, name="t2")   # t2 ~ max(x1, x4)
qp.binary_var(name="b")                                       # b=0 -> branch A, b=1 -> branch B

# (A + t1) + b * ((B + t2) - (A + t1))
qp.minimize(
    constant=A,
    linear={"t1": 1, "b": B - A},
    quadratic={("b", "t2"): 1, ("b", "t1"): -1},
)

qp.linear_constraint({"t1": 1}, ">=", x2, "t1_ge_x2")
qp.linear_constraint({"t1": 1}, ">=", x3, "t1_ge_x3")
qp.linear_constraint({"t2": 1}, ">=", x1, "t2_ge_x1")
qp.linear_constraint({"t2": 1}, ">=", x4, "t2_ge_x4")

conv = QuadraticProgramToQubo(penalty=M)
qubo = conv.convert(qp)
print("qubits:", qubo.get_num_vars())

op, offset = qubo.to_ising()
print(op)
print("offset:", offset)

# exact check: Hamiltonian is diagonal, take its minimum
n = qubo.get_num_vars()
diag = np.real(op.to_matrix(sparse=True).diagonal()) + offset
idx = int(np.argmin(diag))
bits = np.array([(idx >> k) & 1 for k in range(n)])   # Qiskit is little-endian
sol = conv.interpret(bits)                            # [t1, t2, b]

f_true = min(x1 + x4 + max(x3, x2), x3 + x2 + max(x1, x4))
print("ground energy:", diag[idx], "| decoded (t1, t2, b):", sol, "| true f:", f_true)
# The t of the unselected branch is unconstrained by the objective, so it may be > the true max.

qubits: 20
SparsePauliOp(['IIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIZIIIIIIII', 'IIIIIIIIIIZIIIIIIIII', 'IIIIIIIIIZIIIIIIIIII', 'IIIIIIIIZIIIIIIIIIII', 'IIIIIIIZIIIIIIIIIIII', 'IIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIZIIIII', 'IIIIIIIIIIIIIZIIIIII', 'IIIIIIIIIIIIZIIIIIII', 'IIIIIIZIIIIIIIIIIIII', 'IIIIIZIIIIIIIIIIIIII', 'IIIIZIIIIIIIIIIIIIII', 'IIIZIIIIIIIIIIIIIIII', 'IIZIIIIIIIIIIIIIIIII', 'IZIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIZIZ', 'IIIIIIIIIIIIIIIIZIIZ', 'IIIIIIIIIIIZIIIIIIIZ', 'IIIIIIIIIIZIIIIIIIIZ', 'IIIIIIIIIZIIIIIIIIIZ', 'IIIIIIIIZIIIIIIIIIIZ', 'IIIIIIIZIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIZZI', 'IIIIIIIIIIIIIIIIZIZI', 'IIIIIIIIIIIZIIIIIIZI', 'IIIIIIIIIIZIIIIIIIZI', 'IIIIIIIIIZIIIIIIIIZI', 'IIIIIIIIZIIIIIIIIIZI', 'IIIIIIIZIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIZZII', 'IIIIIIIIIIIZIIIIIZII', 'IIIIIIIIIIZIIIIIIZII', 'IIIIIIIIIZIIIIIIIZII', 'IIIIIIIIZIIIIIIIIZII', 'IIIIIIIZIIIII